In [5]:
import numpy as np
import pandas as pd
 # Splitting the data to calculate accuracy based on r2 score
def train_val_split(X, y, val_ratio=0.2, random_state=42):
    np.random.seed(random_state)

    m = X.shape[0]
    indices = np.random.permutation(m)

    val_size = int(m * val_ratio)
    val_idx = indices[:val_size]
    train_idx = indices[val_size:]

    X_train = X[train_idx]
    y_train = y[train_idx]
    X_val = X[val_idx]
    y_val = y[val_idx]

    return X_train, X_val, y_train, y_val

def load_train_data(path):
    df = pd.read_csv(path)
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    return X, y

def load_test_data(path):
    df = pd.read_csv(path)
    return df.values

def normalize_train(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma # z-score normalization
    return X_norm, mu, sigma

def normalize_test(X, mu, sigma):
    return (X - mu) / sigma

def compute_cost(X, y, w, b):
    m = X.shape[0]
    predictions = X @ w + b #Used matrix multplication to reduce run time
    errors = predictions - y
    return (errors @ errors) / (2 * m)

def compute_gradient(X, y, w, b):
    m = X.shape[0]
    predictions = X @ w + b
    errors = predictions - y

    dj_dw = (X.T @ errors) / m
    dj_db = np.sum(errors) / m

    return dj_db, dj_dw

def gradient_descent(X, y, learning_rate=0.01, num_iters=1000):
    X, mu, sigma = normalize_train(X)

    w = np.zeros(X.shape[1])
    b = 0.0
    J_history = []

    for i in range(num_iters):
        dj_db, dj_dw = compute_gradient(X, y, w, b)

        w -= learning_rate * dj_dw
        b -= learning_rate * dj_db

        if i % 50 == 0:
            cost = compute_cost(X, y, w, b)
            J_history.append(cost)
            print(f"Iteration {i:4d}: Cost {cost:.4f}")

    return w, b, mu, sigma, J_history

def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - ss_res / ss_tot

def predict(X, w, b, mu, sigma):
    X = normalize_test(X, mu, sigma)
    return X @ w + b

#To run the model and calculate r2 score
X, y = load_train_data("Linear Regression Train.csv")
X_train, X_val, y_train, y_val = train_val_split(X, y, val_ratio=0.2)
w, b, mu, sigma, J_history = gradient_descent(X_train, y_train,learning_rate=0.01,num_iters=1000)
y_train_pred = predict(X_train, w, b, mu, sigma)
y_val_pred = predict(X_val, w, b, mu, sigma)
print(f"Val R2: {r2_score(y_val, y_val_pred):.4f}")



Iteration    0: Cost 5660.7301
Iteration   50: Cost 1674.5428
Iteration  100: Cost 692.3432
Iteration  150: Cost 352.1845
Iteration  200: Cost 225.1278
Iteration  250: Cost 175.6991
Iteration  300: Cost 155.1534
Iteration  350: Cost 145.5423
Iteration  400: Cost 140.1965
Iteration  450: Cost 136.6070
Iteration  500: Cost 133.8094
Iteration  550: Cost 131.4180
Iteration  600: Cost 129.2678
Iteration  650: Cost 127.2814
Iteration  700: Cost 125.4173
Iteration  750: Cost 123.6507
Iteration  800: Cost 121.9652
Iteration  850: Cost 120.3491
Iteration  900: Cost 118.7938
Iteration  950: Cost 117.2927
Val R2: 0.9406
